# IV. Продвинутые темы — задачки

По каждому разделу занятия "IV. Продвинутые темы" — по 2 задачи для самостоятельной практики.

In [ ]:
!pip install "z3-solver"
from z3 import *

## 1. Выражения, сорта и декларации

### 1.1 Разбор составного выражения

Дано выражение `n = (x + y) * 2 - z`. Убедитесь, что это приложение (`is_app`),
выведите его декларацию, её `kind()`, количество аргументов и сам каждый
аргумент вместе с его сортом.

In [ ]:
x, y, z = Ints('x y z')
n = (x + y) * 2 - z

print("is expression:", is_expr(n))
print("is application:", is_app(n))
print("decl:", n.decl(), " kind:", n.decl().kind())
print("num args:", n.num_args())
for i in range(n.num_args()):
    print("arg(", i, ") ->", n.arg(i), " sort:", n.arg(i).sort())

### 1.2 Подстановка подвыражений

Для двух функций `f(Int, Int) -> Int` и `g(Int) -> Int` постройте выражение
`n = f(g(x), g(g(y)))`, а затем с помощью `substitute` замените `g(x)` на `y`,
а `g(g(y))` на `x + 1`. Выведите исходное и упрощённое выражения.

In [ ]:
x, y = Ints('x y')
f = Function('f', IntSort(), IntSort(), IntSort())
g = Function('g', IntSort(), IntSort())

n = f(g(x), g(g(y)))
print(n)
print(substitute(n, (g(x), y), (g(g(y)), x + 1)))

## 2. Массивы

### 2.1 Коммутативность записи по разным индексам

Докажите, что если `i != j`, то порядок двух последовательных операций
`Store` по разным индексам не важен: `Store(Store(A, i, v1), j, v2)` равно
`Store(Store(A, j, v2), i, v1)`.

In [ ]:
A = Array('A', IntSort(), IntSort())
i, j, v1, v2 = Ints('i j v1 v2')

prove(Implies(i != j,
              Store(Store(A, i, v1), j, v2) == Store(Store(A, j, v2), i, v1)))

### 2.2 Перевод денег между счетами

Смоделируйте банковские счета как массив `Bal: Int -> Int` (индекс — номер
счёта, значение — баланс). "Перевод" суммы `amount` со счёта `acc1` на счёт
`acc2` — это две операции `Store`. Докажите, что суммарный баланс этих двух
счетов после перевода не меняется (при `acc1 != acc2`).

In [ ]:
Bal = Array('Bal', IntSort(), IntSort())
acc1, acc2, amount = Ints('acc1 acc2 amount')

Bal2 = Store(Bal, acc1, Bal[acc1] - amount)
Bal3 = Store(Bal2, acc2, Bal2[acc2] + amount)

prove(Implies(acc1 != acc2,
              Bal[acc1] + Bal[acc2] == Bal3[acc1] + Bal3[acc2]))

## 3. Кванторы

### 3.1 Поиск функции по количественным условиям

Найдите (с помощью `solve`) функцию `f(x, y)`, такую что она симметрична
(`ForAll([x, y], f(x, y) == f(y, x))`) и при этом существует такое `x`, что
`f(x, x) == 5`.

In [ ]:
f = Function('f', IntSort(), IntSort(), IntSort())
x, y = Ints('x y')

solve(ForAll([x, y], f(x, y) == f(y, x)),
      Exists(x, f(x, x) == 5))

### 3.2 Индуктивное доказательство суммы квадратов

Докажите по аналогии с примером про $\sum_{i=0}^n i = \frac{n(n+1)}{2}$,
что $\sum_{i=0}^n i^2 = \frac{n(n+1)(2n+1)}{6}$. Определите функцию `SumSq`
рекуррентно и используйте вспомогательную функцию `induction`, чтобы
подсказать солверу схему индукции.

In [ ]:
n = Int('n')
SumSq = Function('sumsq', IntSort(), IntSort())
s = Solver()
s.add(SumSq(0) == 0)
s.add(ForAll([n], SumSq(n + 1) == (n + 1) * (n + 1) + SumSq(n)))

def indSumSq(n):
    return 6 * SumSq(n) == n * (n + 1) * (2 * n + 1)

def induction(p):
    n = Int('n')
    return Implies(And(p(0),
                        ForAll([n], Implies(And(n >= 0, p(n)), p(n + 1)))),
                   ForAll([n], Implies(n >= 0, p(n))))

s.add(induction(indSumSq))
s.add(Not(ForAll([n], Implies(n >= 0, indSumSq(n)))))  # пытаемся найти контрпример
print(s.check())  # unsat -> формула доказана

## 4. Оптимизация

### 4.1 Максимизация прибыли (линейная задача)

Фабрика производит товары A и B в количествах `x` и `y`. На единицу товара A
уходит 2 единицы ресурса 1 и 1 единица ресурса 2, на единицу товара B — 1
единица ресурса 1 и 3 единицы ресурса 2. Всего доступно 100 единиц ресурса 1
и 120 единиц ресурса 2. Прибыль с единицы A равна 3, с единицы B — 5.
Найдите `x` и `y`, максимизирующие прибыль.

In [ ]:
x, y = Reals('x y')
opt = Optimize()
opt.add(x >= 0, y >= 0)
opt.add(2 * x + y <= 100)   # ограничение по ресурсу 1
opt.add(x + 3 * y <= 120)   # ограничение по ресурсу 2

profit = 3 * x + 5 * y
opt.maximize(profit)

print(opt.check())
print(opt.model())

### 4.2 Коробка максимального объёма (нелинейная задача)

Из прямоугольного листа картона 20x30 по углам вырезают квадраты со стороной
`t`, после чего боковые части загибают вверх и получают открытую коробку.
Найдите `t` (0 < t < 10), максимизирующее объём коробки
`(20 - 2t)(30 - 2t)t`. Обратите внимание на `reason_unknown()` — оптимизатор
не всегда гарантированно находит точный оптимум в нелинейных задачах.

In [ ]:
t = Real('t')
opt = Optimize()
opt.add(t > 0, t < 10)

volume = (20 - 2 * t) * (30 - 2 * t) * t
opt.maximize(volume)

print(opt.check())
print(opt.model())
print(opt.reason_unknown())

## 5. Множественные солверы

### 5.1 Общий базовый солвер и два независимых расширения

Создайте базовый солвер `base` с условиями `x + y == 20, x > 0, y > 0`.
Скопируйте его условия в два новых солвера `s1` и `s2`: в `s1` добавьте
условие `x > y`, а в `s2` — условие `x == y`. Проверьте выполнимость каждого
и выведите модели.

In [ ]:
x, y = Ints('x y')
base = Solver()
base.add(x + y == 20, x > 0, y > 0)

s1 = Solver()
s1.add(base.assertions())
s1.add(x > y)
print("s1:", s1.check(), s1.model() if s1.check() == sat else None)

s2 = Solver()
s2.add(base.assertions())
s2.add(x == y)
print("s2:", s2.check(), s2.model() if s2.check() == sat else None)

### 5.2 Проверка нескольких гипотез через push/pop

На одном солвере с базовыми условиями `x >= 0, y >= 0, x + y == 10` по
очереди проверьте с помощью `push`/`pop` две гипотезы: `x - y == 4` и
заведомо невыполнимую `x - y == 100`. Убедитесь, что после каждого `pop`
солвер возвращается к исходному, базовому набору условий.

In [ ]:
s = Solver()
s.add(x >= 0, y >= 0, x + y == 10)

s.push()
s.add(x - y == 4)
print("гипотеза x - y == 4:", s.check(), s.model())
s.pop()

s.push()
s.add(x - y == 100)
print("гипотеза x - y == 100:", s.check())
s.pop()

print("солвер после pop снова базовый:", s.check(), s.model())